# 02 - Exploratory Data Analysis (EDA)

This notebook performs comprehensive exploratory analysis.

## Objectives:
- Identify missing values and outliers
- Analyze sales trends
- Examine product and store performance
- Study seasonality patterns
- Analyze holiday impact

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from src.data_loader import DataLoader
from src.preprocessing import DataPreprocessor

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('default')
sns.set_palette('husl')

In [ ]:
loader = DataLoader(data_dir='../data/raw')
calendar_df = loader.load_calendar()
sales_df = loader.load_sales()
prices_df = loader.load_prices()

print("Data loaded successfully!")

## 1. Missing Values Analysis

In [ ]:
print("Missing Values in Calendar:")
print(calendar_df.isnull().sum())

print("\nMissing Values in Sales:")
print(sales_df.isnull().sum().sum())

print("\nMissing Values in Prices:")
print(prices_df.isnull().sum())

## 2. Sales Trends Analysis

In [ ]:
preprocessor = DataPreprocessor()
sales_long = preprocessor.melt_sales_data(sales_df)
sales_long = sales_long.merge(calendar_df[['d', 'date']], on='d')

print(f"Sales data melted: {sales_long.shape}")
sales_long.head()

In [ ]:
daily_sales = sales_long.groupby('date')['sales'].sum().reset_index()

fig = px.line(daily_sales, x='date', y='sales', title='Total Daily Sales Over Time')
fig.show()

print(f"\nTotal sales: {daily_sales['sales'].sum():,.0f}")
print(f"Average daily sales: {daily_sales['sales'].mean():,.0f}")

## 3. Category Performance

In [ ]:
category_sales = sales_long.groupby('cat_id')['sales'].sum().sort_values(ascending=False)

fig = px.bar(x=category_sales.index, y=category_sales.values,
            title='Total Sales by Category',
            labels={'x': 'Category', 'y': 'Total Sales'})
fig.show()

print("\nCategory Sales:")
print(category_sales)

## 4. Store Performance

In [ ]:
store_sales = sales_long.groupby('store_id')['sales'].sum().sort_values(ascending=False)

fig = px.bar(x=store_sales.index, y=store_sales.values,
            title='Total Sales by Store',
            labels={'x': 'Store', 'y': 'Total Sales'},
            color=store_sales.values)
fig.show()

print("\nStore Sales:")
print(store_sales)

## 5. Seasonality Analysis

In [ ]:
sales_long['month'] = sales_long['date'].dt.month
sales_long['year'] = sales_long['date'].dt.year

monthly_sales = sales_long.groupby(['year', 'month'])['sales'].sum().reset_index()
monthly_sales['year_month'] = pd.to_datetime(monthly_sales[['year', 'month']].assign(day=1))

fig = px.line(monthly_sales, x='year_month', y='sales',
             title='Monthly Sales Pattern',
             labels={'year_month': 'Month', 'sales': 'Total Sales'})
fig.show()

In [ ]:
sales_long['dayofweek'] = sales_long['date'].dt.dayofweek
dow_sales = sales_long.groupby('dayofweek')['sales'].mean()
dow_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig = px.bar(x=dow_names, y=dow_sales.values,
            title='Average Sales by Day of Week',
            labels={'x': 'Day of Week', 'y': 'Average Sales'})
fig.show()

## 6. Holiday Impact Analysis

In [ ]:
sales_with_events = sales_long.merge(calendar_df[['d', 'event_name_1', 'event_type_1']], on='d')
sales_with_events['has_event'] = (sales_with_events['event_name_1'] != 'None').astype(int)

event_comparison = sales_with_events.groupby('has_event')['sales'].mean()

fig = px.bar(x=['No Event', 'Event'], y=event_comparison.values,
            title='Average Sales: Events vs Non-Events',
            labels={'x': '', 'y': 'Average Sales'})
fig.show()

print(f"\nAverage sales with events: {event_comparison[1]:.2f}")
print(f"Average sales without events: {event_comparison[0]:.2f}")
print(f"Event uplift: {(event_comparison[1]/event_comparison[0] - 1)*100:.1f}%")

## 7. Top Products Analysis

In [ ]:
product_sales = sales_long.groupby('item_id')['sales'].sum().sort_values(ascending=False).head(20)

fig = px.bar(x=product_sales.index, y=product_sales.values,
            title='Top 20 Products by Total Sales',
            labels={'x': 'Product ID', 'y': 'Total Sales'},
            color=product_sales.values)
fig.show()

## 8. Price Distribution

In [ ]:
fig = px.histogram(prices_df, x='sell_price', nbins=50,
                  title='Price Distribution',
                  labels={'sell_price': 'Price ($)'})
fig.show()

print("\nPrice Statistics:")
print(prices_df['sell_price'].describe())

## 9. Outlier Detection

In [ ]:
Q1 = sales_long['sales'].quantile(0.25)
Q3 = sales_long['sales'].quantile(0.75)
IQR = Q3 - Q1

outliers = sales_long[(sales_long['sales'] < Q1 - 1.5 * IQR) | 
                     (sales_long['sales'] > Q3 + 1.5 * IQR)]

print(f"Number of outliers: {len(outliers)} ({len(outliers)/len(sales_long)*100:.2f}%)")
print(f"\nOutlier statistics:")
print(outliers['sales'].describe())

## Summary of Key Findings

1. **Sales Trends**: Identify overall patterns and growth
2. **Seasonality**: Strong monthly and weekly patterns exist
3. **Events**: Positive impact on sales
4. **Store Performance**: Varies significantly across locations
5. **Category Performance**: Some categories dominate sales
6. **Outliers**: Small percentage but worth investigating